# Practical 7: Data Wrangling for Aggregated Analysis

**Objective:** Aggregate and reshape data for deeper insights.

- Group by IP to analyze frequency of attacks
- Time-series resampling (hourly/daily traffic)
- Pivot tables to see request type by label
- Filter out bots and internal IPs

**Note on source data:** this practical uses `logs/features_access_log.csv` (Practical 5's output) rather than `logs/balanced_access_log.csv` (Practical 6's output). The balanced dataset drops `IP Address` and `Date/Time` before SMOTE, since those identifier columns can't meaningfully carry over to synthetic rows. Aggregation by IP and time only makes sense on real, non-synthetic records.

In [3]:
import pandas as pd

df = pd.read_csv('logs/features_access_log.csv', parse_dates=['Date/Time'])
print('Rows loaded:', len(df))
df.head()

Rows loaded: 47685


,IP Address,Date/Time,Request Type,Resource,Status Code,requests_per_ip,time_since_prev_request,is_error_status,error_rate_per_ip,ua_type,url_entropy,label
0,1.0.170.238,2026-08-05 17:39:50+05:30,GET,/products,200,1,9999.0,0,0.0,browser,3.169925,benign
1,1.100.161.254,2026-08-02 09:28:10+05:30,GET,/help,200,1,9999.0,0,0.0,browser,2.321928,benign
2,1.104.218.20,2026-08-14 05:02:08+05:30,GET,/about,200,1,9999.0,0,0.0,browser,2.584963,benign
3,1.105.34.107,2026-08-10 03:12:08+05:30,GET,/products?id=1',500,1,9999.0,1,1.0,browser,3.773557,sqli
4,1.106.111.1,2026-08-02 11:30:41+05:30,GET,/search?q=laptop,200,1,9999.0,0,0.0,browser,3.750000,benign


## 1. Group by IP: frequency of attacks per source

In [4]:
attacks_only = df[df['label'] != 'benign']
attacks_per_ip = attacks_only.groupby('IP Address')['label'].value_counts().unstack(fill_value=0)
attacks_per_ip['total_attacks'] = attacks_per_ip.sum(axis=1)
attacks_per_ip.sort_values('total_attacks', ascending=False).head(10)

label,brute_force,path_traversal,sqli,total_attacks
IP Address,,,,
139.11.49.144,25,0,0,25
221.82.246.172,25,0,0,25
56.7.5.9,25,0,0,25
93.89.37.44,25,0,0,25
41.203.107.144,25,0,0,25
201.126.226.82,25,0,0,25
162.128.201.25,24,0,0,24
151.116.253.29,24,0,0,24
160.49.247.216,24,0,0,24


## 2. Time-series resampling

Resample the full traffic volume into hourly and daily buckets to see traffic shape over time.

In [5]:
ts = df.set_index('Date/Time')

hourly_traffic = ts.resample('h').size()
daily_traffic = ts.resample('D').size()

print('Hourly traffic (first 10 buckets):')
print(hourly_traffic.head(10))
print('\nDaily traffic:')
print(daily_traffic)

Hourly traffic (first 10 buckets):
Date/Time
2026-08-01 00:00:00+05:30    133
2026-08-01 01:00:00+05:30    124
2026-08-01 02:00:00+05:30    147
2026-08-01 03:00:00+05:30    121
2026-08-01 04:00:00+05:30    159
2026-08-01 05:00:00+05:30    157
2026-08-01 06:00:00+05:30    152
2026-08-01 07:00:00+05:30    166
2026-08-01 08:00:00+05:30    127
2026-08-01 09:00:00+05:30    133
Freq: h, dtype: int64

Daily traffic:
Date/Time
2026-08-01 00:00:00+05:30    3395
2026-08-02 00:00:00+05:30    3348
2026-08-03 00:00:00+05:30    3376
2026-08-04 00:00:00+05:30    3417
2026-08-05 00:00:00+05:30    3324
2026-08-06 00:00:00+05:30    3313
2026-08-07 00:00:00+05:30    3519
2026-08-08 00:00:00+05:30    3408
2026-08-09 00:00:00+05:30    3363
2026-08-10 00:00:00+05:30    3389
2026-08-11 00:00:00+05:30    3471
2026-08-12 00:00:00+05:30    3401
2026-08-13 00:00:00+05:30    3476
2026-08-14 00:00:00+05:30    3485
Freq: D, dtype: int64


In [6]:
# Same resampling, but split by label, to see whether attack traffic clusters
# at particular hours/days rather than spreading evenly like benign traffic.
hourly_by_label = ts.groupby('label').resample('h').size().unstack('label', fill_value=0)
hourly_by_label.head(10)

label,benign,brute_force,path_traversal,sqli
Date/Time,,,,
2026-08-01 00:00:00+05:30,126,0,5,2
2026-08-01 01:00:00+05:30,120,0,0,4
2026-08-01 02:00:00+05:30,141,0,2,4
2026-08-01 03:00:00+05:30,118,0,2,1
2026-08-01 04:00:00+05:30,128,23,5,3
2026-08-01 05:00:00+05:30,122,31,1,3
2026-08-01 06:00:00+05:30,150,0,1,1
2026-08-01 07:00:00+05:30,146,16,3,1
2026-08-01 08:00:00+05:30,121,0,4,2


## 3. Pivot table: request type by label

In [7]:
pivot_request_label = pd.pivot_table(
    df, index='Request Type', columns='label',
    values='Status Code', aggfunc='count', fill_value=0
)
pivot_request_label

label,benign,brute_force,path_traversal,sqli
Request Type,,,,
GET,39296,0,1000,967
POST,3937,2485,0,0


## 4. Filter out bots and internal IPs

Internal/private IPs (`192.168.x.x`) represent traffic from inside the network, not external visitors or attackers. Bot traffic (`ua_type == 'bot'`) is a separate category worth analyzing on its own rather than mixed into general human/browser traffic. Filtering both out isolates genuine external browser traffic.

In [8]:
is_internal_ip = df['IP Address'].str.startswith('192.168.')
is_bot = df['ua_type'] == 'bot'

external_human_traffic = df[~is_internal_ip & ~is_bot]

print(f'Total rows: {len(df)}')
print(f'Internal IP rows: {is_internal_ip.sum()}')
print(f'Bot rows: {is_bot.sum()}')
print(f'Remaining external human traffic: {len(external_human_traffic)}')

external_human_traffic['label'].value_counts()

Total rows: 47685
Internal IP rows: 2015
Bot rows: 5347
Remaining external human traffic: 40323


label
benign            38088
brute_force        1226
path_traversal      512
sqli                497
Name: count, dtype: int64

## Save aggregated outputs for Practical 8

In [9]:
attacks_per_ip.to_csv('logs/attacks_per_ip.csv')
hourly_by_label.to_csv('logs/hourly_traffic_by_label.csv')
external_human_traffic.to_csv('logs/external_human_traffic.csv', index=False)
print('Saved: logs/attacks_per_ip.csv, logs/hourly_traffic_by_label.csv, logs/external_human_traffic.csv')

Saved: logs/attacks_per_ip.csv, logs/hourly_traffic_by_label.csv, logs/external_human_traffic.csv
